In [0]:
%run ../configs/config



In [0]:
spark.sql('use catalog banking_project')

In [0]:
from pyspark.sql import functions as F

In [0]:
accounts_df = f'{bronze_schema}.accounts'
accounts_read_df = spark.table(accounts_df)

In [0]:
accounts_dropped_df = accounts_read_df.dropDuplicates()

In [0]:

accounts_null_df = accounts_dropped_df.filter(
    F.col('account_id').isNotNull()
)


In [0]:
accounts_translate_df = (accounts_null_df
                         .withColumn('frequency',
                            F.when(F.col('frequency') == 'POPLATEK MESICNE','Monthly')
                            .when(F.col('frequency') == 'POPLATEK TYDNE', 'Weekly')
                            .when(F.col('frequency') == 'POPLATEK PO OBRATU',
                                'After Each Transaction')
                            .otherwise('Unknown')
)
                         
    ).withColumn('date', F.to_date(F.concat(F.lit('19'), F.col('date').cast('string')), 'yyyyMMdd')
        ).withColumnRenamed('date', 'opened_date')

In [0]:
accounts_final_df = add_metadata_columns(accounts_translate_df)

In [0]:
(
    accounts_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{silver_schema}.accounts')
)

In [0]:
display(spark.table(f'{silver_schema}.accounts'))